In [1]:
#Imports
import numpy as np
import h5py
import os
import re

## Section 0, user input

In [ ]:
#User input
title='Example XRD measurement' #a title for the measurement
filename="60C_air_15_1ass1_axssoll_035sci_2tho_1_normalized.uxd"
output="XRD_CNR-IMM@CT_output.nxs"
measurement_type='XRD' #'XRD' or 'XRR'

#"Dome" information, must be inserted manually. Leave blank if dome is not used.
dome_temperature=''
dome_temperature_units='C' #C or K
dome_atmosphere='' #{air, vacuum, nitrogen...}

#SAMPLE NAME AND IDENTIFIER
#OPTIONAL UNLESS FOR NFFA-DI
sample_name=''
sample_identifier=''

#COMMENT
#OPTIONAL. If left blank, the measurement file's comment lines will be used 
comment=''

## Section 1, read UXD file

In [3]:
#Open and read UXD file
def readUXD(filename):
    """
    Reads the contents of a UXD file and outputs:
    -a dictionary metadata with the metadata contained in the header (raw, unprocessed for additional inferred information)
    -an array data with the actual measurement data, usually composed of two columns
    -a list of string comments with the comment lines
    """
    
    ''' Comment on how to read
    The type of line is determined by the first character, specified in the manual
    _ starting character are dictionary-like metadata, key=value pairs
    + - . or digit indicates numerical data
    ; indicates a comment
    blank line has no meaning (empty comment)
    '''
    if filename[-4:] != '.uxd':
        raise ValueError('File extention is not .uxd')
    file=open(filename)
    

    metadata={}
    data_rows=[] #as a list, then converted into ndarray
    comments=[]

    for line in file:
        if line[0] == ';':
            comments.append(line)
        elif line.strip() == '':
            pass
        elif line.strip()[0] in ['+','-', '.', '1', '2', '3', '4', '5', '6', '7', '8', '9']:
            data_rows.append(line.strip().split())
        elif line[0] == '_':
            split=line[1:].split('=')
            metadata[split[0].strip()] = split[1].strip()
        else:
            raise ValueError(f'line {line} breaks expected format')
    file.close()
    data = np.array(data_rows, dtype=float)

    return metadata, data, comments
    
meta, data, filecomments = readUXD(filename)

In [ ]:

#A few consistency checks
if meta.get('FILEVERSION') != '2':
    print('WARNING. UXD fileversion different than expected or not found. Parsing may work but be wrong.')

if (meta.get('2THETACOUNTS') or meta.get('2THETACPS'))  and (float(meta['2THETA']) != round(data[0,0],1)):
    raise ValueError('Inconsistency between metadata and data (initial 2theta value)')

if (meta['SITE'] != 'Italy') or (meta['USER'] != 'Administrator'):
    print('WARNING. Unexpected site/user metadata')

if meta['GONIOMETER_CODE'] != '20':
    print('WARNING. Unexpected goniometer metadata')

if meta['STAGE_CODE'] != '13':
    print('WARNING. Unexpected stage metadata')

#detector conditions
detector_conditions=[
    meta['DETECTOR'] != '1',
    meta['HV'] != '665',
    meta['GAIN'] != '80',
    meta['LLD'] != '0.592',
    meta['ULD'] != '1.474',
    ]
if any(detector_conditions):
    print('WARNING. Unexpected detector metadata')

#source conditions
source_conditions=[
    meta['WL1'] != '1.5406',
    meta['WL2'] != '1.54439',
    meta['WL3'] != '1.39222',
    meta['WLRATIO'] != '0.5',
    ]
if any(source_conditions):
    print('WARNING. Unexpected source metadata')

In [ ]:
#auxiliary code for filename parsing

#split filename in optical and nonoptical parts
#doesn't really make sense to do it here by itself, but it works and it's not time for cleanup.

#Split filename, stripped of the extension
filename_contents=filename[0:-4].split('_')

#list of regular expressions that may match optical elements
optical_regexes=[
    '[0-9.]*-?(ass|abs)[0-9.]*',    #matches e.g. 1ass1 and some variants I've seen.
    'axssoll',   #literal
    '035sci',   #literal
    '(?i)^ge\d{3}$', #e.g. Ge220, Ge044; should always be the last source side before sample
]

patterns = [re.compile(rx) for rx in optical_regexes]

optical_elements=[]
nonoptical_filename=[]

for s in filename_contents:
    if any(p.match(s) for p in patterns):
        optical_elements.append(s)
    else:
        nonoptical_filename.append(s)

## Section 2, write into a NeXus file

In [ ]:
#NeXus writing block 1, create file and write into the root /entry group fields.
try: f.close()
except NameError: pass

f=h5py.File(output, 'w')

f.create_group('/entry')
f['/entry'].attrs['NX_class']='NXentry'
f['/entry'].attrs['default']='data'
f["/entry/definition"]="NXxrd_bru"
f["/entry/definition"].attrs["version"]="2025-12-03"
#f["/entry/definition"].attrs["url"]="insert github permalink here"

f['/entry'].attrs['default']='data'

if comment:
    f["/entry/comment"] = comment
elif filecomments:
    f["/entry/comment"] = ''.join(filecomments)
if title:
    f["/entry/title"] = title
if meta['DATEMEASURED']:
    date_parts = meta['DATEMEASURED'].strip('"').split(" ")
    date=date_parts[0].split('-')
    time=date_parts[1]
    day = date[0].zfill(2)
    month = {
        "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
        "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
        "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
    }[date[1]]
    year = date[2]
    f["/entry/start_time"] = f"{year}-{month}-{day}T{time}"

#original filename:
try:
    prefixlength = 19
    suffixlength = 2
    path = filecomments[0][prefixlength:-suffixlength]
    original_filename = os.path.basename(path)
    name, ext = os.path.splitext(original_filename)
    if ext.lower() == ".raw" and name:
        #valid original filename, write to nexus field
        f["/entry/data_file"] = original_filename
    if filename[0:-4] != original_filename[0:-4]:
        choice = input("Warning! Current and original filename do not match. Continue? (Y/any): ")
        if choice.lower() == 'y':
            pass
        else:
            raise SystemExit("Script stopped by user.")
except: IndexError 


#XRD or XRR detection:
#LOGIC REMOVED, user input instead

if measurement_type == 'XRD':
    f["/entry/measurement_type"]='XRD'
elif measurement_type == 'XRR':
    f["/entry/measurement_type"]='XRR'
else:
    raise ValueError


f["/entry/experiment_location"] = "CNR - IMM Catania"

Warning! XRD experiment type undetected


In [ ]:
#sample block
f.create_group('/entry/sample')
f['/entry/sample'].attrs['NX_class'] = 'NXsample'

if sample_identifier:
    f["/entry/sample/identifier"] = sample_identifier

if sample_name:
    f["/entry/sample/name"] = sample_name
    if meta['SAMPLE']:
        f["/entry/sample/name_fromfile"] = meta['SAMPLE']+meta.get('+SAMPLE')
elif meta['SAMPLE']:
    f["/entry/sample/name"] = meta['SAMPLE']+meta.get('+SAMPLE')

if dome_temperature or dome_atmosphere:
    f.create_group('/entry/sample/environment')
    f['/entry/sample/environment'].attrs['NX_class'] = 'NXenvironment'
    if dome_temperature:
        f['/entry/sample/environment/temperature'] = float(dome_temperature)
        if dome_temperature_units == 'C':
            f['/entry/sample/environment/temperature'].attrs['units'] = 'C'
        elif dome_temperature_units == 'K':
            f['/entry/sample/environment/temperature'].attrs['units'] = 'K'
        else:
            raise ValueError
    else:
        f['/entry/sample/environment/temperature'] = 20
        f['/entry/sample/environment/temperature'].attrs['units'] = 'C'

    if dome_atmosphere:
        f['/entry/sample/environment/atmosphere'] = dome_atmosphere
    else:
        f['/entry/sample/environment/atmosphere'] = 'air'

In [ ]:
#data block
f.create_group('/entry/data')
f['/entry/data'].attrs['NX_class']='NXdata'

#links from instrument/detector/data; links must be defined after their target.

In [ ]:
#instrument block 1 (probably gonna split certain subgroups)

f.create_group('/entry/instrument')
f['/entry/instrument'].attrs['NX_class']='NXinstrument'
f['/entry/instrument/name']='Bruker AXS - D8 Discover'


#beam (redundant imho but hey...)
f.create_group('/entry/instrument/beam')
f['/entry/instrument/beam'].attrs['NX_class']='NXbeam'
def angstrom_to_keV(wavelength_angstrom):
    return 12.398 / wavelength_angstrom
f['/entry/instrument/beam/incident_energy'] = angstrom_to_keV(float(meta['WL1']))
f['/entry/instrument/beam/incident_energy'].attrs['units'] = "keV"

#source
f.create_group('/entry/instrument/source')
f['/entry/instrument/source'].attrs['NX_class']='NXsource'
f['/entry/instrument/source/type']="Fixed tube X-ray"
f['/entry/instrument/source/xray_tube_material']=meta['ANODE']
f['/entry/instrument/source/power'] = 3
f['/entry/instrument/source/power'].attrs['units'] = 'kW'
#potenza 3kW

f['/entry/instrument/source/name']="KFL CU 2 K nr. 400903"
f['/entry/instrument/source/probe']="x-ray"

f['/entry/instrument/source/source_peak_wavelength']=float(meta['WL1'])
f['/entry/instrument/source/source_peak_wavelength'].attrs['units']='Angstrom'

f['/entry/instrument/source/wavelength_alpha_one'] = float(meta['WL1'])
f['/entry/instrument/source/wavelength_alpha_one'].attrs['units']='Angstrom'

f['/entry/instrument/source/wavelength_alpha_two'] = float(meta['WL2'])
f['/entry/instrument/source/wavelength_alpha_two'].attrs['units']='Angstrom'

f['/entry/instrument/source/wavelength_beta'] = float(meta['WL3'])
f['/entry/instrument/source/wavelength_beta'].attrs['units']='Angstrom'

f['/entry/instrument/source/ratio_alphatwo_alphaone'] = float(meta['WLRATIO'])

#little consistency check
if float(meta['WL1']) != 1.5406:
    print("Warning! wavelength different than expected")
if meta['WL_UNIT'] != 'A':
    print("Warning! Wavelength units different than expected. Expected Angstrom")


In [ ]:

#little utility function

def parse_number(s: str) -> float:
    # Case: explicit decimal
    if "." in s:
        return float(s)
    # Case: starts with '.' (like ".1")
    if s.startswith("."):
        return float("0" + s)
    # Case: leading zero without decimal (like "01")
    if s.startswith("0") and len(s) > 1:
        # interpret as implicit decimal: "01" -> "0.1", "001" -> "0.01"
        return float("0." + s[1:])
    # Default: plain integer string
    return float(s)


In [ ]:
#instrument block 2, source optics
#describes all optics between source and sample

f.create_group('/entry/instrument/source_optics')
f['/entry/instrument/source_optics'].attrs['NX_class']='NXcomponent'
f['/entry/instrument/source_optics/description']='This group describes optics between source and sample'

#monochromator always present
f.create_group('/entry/instrument/source_optics/monochromator')
f['/entry/instrument/source_optics/monochromator'].attrs['NX_class']='NXmonochromator'
optics_sequence_index=1
f['/entry/instrument/source_optics/monochromator/optics_sequence_index']=optics_sequence_index
optics_sequence_index += 1
f['/entry/instrument/source_optics/monochromator/wavelength']=1.5406
f['/entry/instrument/source_optics/monochromator/name']='Göbel mirror GM3'
#...and other details? Vendor, whatever...

presample_regexes=[
    '([0-9.]*)-?(ass|abs)([0-9.]*)',    #matches e.g. 1ass1 and some variants I've seen.
    'axssoll',   #literal
    '(?i)ge\d{3}', #e.g. Ge220, Ge044
]
postsample_regexes=[
    '(\d+)sci',   #number-sci scitillator with number angle in degrees capturing group
]
presample_patterns= [re.compile(rx) for rx in presample_regexes]
postsample_patterns= [re.compile(rx) for rx in postsample_regexes]

for component in optical_elements:
    #check that the component is indeed a pre-sample component
    if any(p.match(component) for p in postsample_patterns):
        break
    
    #case by case, as they command different meaning, parse the components for what they might be
    m = presample_patterns[0].match(component)
    if m:
        pre, middle, post = m.groups()

        f.create_group('/entry/instrument/source_optics/slit1')
        f['/entry/instrument/source_optics/slit1'].attrs['NXclass']='NXslit'
        f['/entry/instrument/source_optics/slit1/description']='Slit placed before the absorber on the source side'
        f['/entry/instrument/source_optics/slit1/optics_sequence_index']=optics_sequence_index
        optics_sequence_index += 1
        f['/entry/instrument/source_optics/slit1/x_gap']=parse_number(pre)
        f['/entry/instrument/source_optics/slit1/x_gap'].attrs['units']='mm'

        f.create_group('/entry/instrument/source_optics/absorber')
        f['/entry/instrument/source_optics/absorber'].attrs['NXclass']='NXattenuator'
        f['/entry/instrument/source_optics/absorber/description']='Absorber on source side; atuomatically adjusted by instrument for signal to noise ratio.'
        f['/entry/instrument/source_optics/absorber/optics_sequence_index']=optics_sequence_index
        optics_sequence_index += 1

        f.create_group('/entry/instrument/source_optics/slit2')
        f['/entry/instrument/source_optics/slit2'].attrs['NXclass']='NXslit'
        f['/entry/instrument/source_optics/slit2/description']='Slit placed after the absorber on the source side'
        f['/entry/instrument/source_optics/slit2/optics_sequence_index']=optics_sequence_index
        optics_sequence_index += 1
        f['/entry/instrument/source_optics/slit2/x_gap']=parse_number(post)
        f['/entry/instrument/source_optics/slit2/x_gap'].attrs['units']='mm'
    
    
    if presample_patterns[1].match(component):
        f.create_group('/entry/instrument/source_optics/soller')
        f['/entry/instrument/source_optics/soller'].attrs['NXclass']='NXcollimator'
        f['/entry/instrument/source_optics/soller/type']='Soller'
        #f['/entry/instrument/source_optics/soller/soller_angle']= #non c'è scritto, so solo che sono "lunghe"
        #f['/entry/instrument/source_optics/soller/soller_angle'].attrs['units'] = 'deg'
        f['/entry/instrument/source_optics/soller/optics_sequence_index']=optics_sequence_index
        optics_sequence_index += 1
    
    if presample_patterns[2].match(component):
        f.create_group('/entry/instrument/source_optics/high_resolution_monochromator')
        f['/entry/instrument/source_optics/high_resolution_monochromator'].attrs['NXclass']='NXmonochromator'
        f['/entry/instrument/source_optics/high_resolution_monochromator/wavelength']=1.5406
        f['/entry/instrument/source_optics/high_resolution_monochromator/optics_sequence_index']=optics_sequence_index
        optics_sequence_index += 1


In [ ]:
#instrument block 3, detector optics, and detector

#describes all optics between sample and detector

f.create_group('/entry/instrument/detector_optics')
f['/entry/instrument/detector_optics'].attrs['NX_class']='NXcomponent'
f['/entry/instrument/detector_optics/description']='This group describes optics between sample and detector'

optics_sequence_index=1 #reset

for component in optical_elements:
    #quickly go through the source optics
    if any(p.match(component) for p in presample_patterns):
        continue
    
    #case by case, as they command different meaning, parse the components for what they might be
    m = postsample_patterns[0].match(component)
    if m:
        sangle = m.groups()
        f.create_group('/entry/instrument/detector_optics/soller')
        f['/entry/instrument/detector_optics/soller'].attrs['NXclass']='NXcollimator'
        f['/entry/instrument/detector_optics/soller/type']='Soller'
        f['/entry/instrument/detector_optics/soller/soller_angle'] = sangle
        f['/entry/instrument/detector_optics/soller/soller_angle'].attrs['units']="deg"
        f['/entry/instrument/detector_optics/soller/optics_sequence_index'] = optics_sequence_index
        optics_sequence_index += 1
        

#fixed detector info
f.create_group('/entry/instrument/detector')
f['/entry/instrument/detector'].attrs['NX_class']='NXdetector'
#f['/entry/instrument/detector/name']=''
#f['/entry/instrument/detector/description']=''
f['/entry/instrument/detector/type']='scintillator'
f['/entry/instrument/detector/vendor']='Bruker'
#f['/entry/instrument/detector/model']=''

#auxiliary part, detect which data type is in the file.
datatype=[]
if meta.get('COUNTS') == '1':
    datatype.append('counts')
if meta.get('cps') == '1':
    datatype.append('cps')
if meta.get('2THETACOUNTS') == '1':
    datatype.append('angle+counts')
if meta.get('2THETACPS') == '1':
    datatype.append('angle+cps')
if len(datatype) != 1:
    raise ValueError('There seem to be either no data, more than one data block, or an unrecognized data block in the file.')

f["/entry/instrument/detector/data_type"] = datatype[0]

#do the rest of the writing based on datatype info
if datatype[0] in ['counts', 'cps']:
    f["/entry/instrument/detector/data"] = data[:, 0]
    if datatype[0] == 'counts':
        f["/entry/instrument/detector/data"].attrs['units'] = 'counts'  
    elif datatype[0] == 'cps':
        f["/entry/instrument/detector/data"].attrs['units'] = 'counts per second'
    else:
        raise ValueError
elif datatype[0] in ['angle+counts', 'angle+cps']:
    f["/entry/instrument/detector"].attrs['axes'] = 'polar_angle'
    f["/entry/instrument/detector"].attrs['signal'] = 'data'
    f["/entry/instrument/detector/polar_angle"] = data[:, 0]
    f["/entry/instrument/detector/polar_angle"].attrs['units'] = 'deg'
    f["/entry/instrument/detector/data"] = data[:, 1]
    if datatype[0] == 'counts':
        f["/entry/instrument/detector/data"].attrs['units'] = 'counts'  
    elif datatype[0] == 'cps':
        f["/entry/instrument/detector/data"].attrs['units'] = 'counts per second'
    else:
        raise ValueError

#links in data group

#LINKS
f['/entry/instrument/detector/data'].attrs['target']='/entry/instrument/detector/data'
f['/entry/data/data'] = f['/entry/instrument/detector/data']
if 'angle' in datatype[0]:
    f['/entry/instrument/detector/polar_angle'].attrs['target']='/entry/instrument/detector/polar_angle'
    f['/entry/data/polar_angle'] = f['/entry/instrument/detector/polar_angle']
    #set default plot if there are polar angle data
    f['/entry/data'].attrs['axes']='polar_angle'
    f['/entry/data'].attrs['signal']='data'

In [13]:
f.close()